# RFM customer segmentation in pure SQL

**SQL (Advanced)** (HackerRank)

> Re-implement the dissertation's RFM (Recency, Frequency, Monetary) customer scoring as a single SQL query instead of a pandas pipeline.

**Concepts:** CTEs, Aggregation, CASE expressions, NTILE / percentile-style bucketing

This notebook is runnable as-is: it creates an in-memory SQLite database with small synthetic sample data, then runs the actual SQL for this project.

In [ ]:
import sqlite3
conn = sqlite3.connect(':memory:')
cur = conn.cursor()
cur.executescript('''
CREATE TABLE orders (customer_id INTEGER, invoice_id INTEGER, invoice_date TEXT, revenue REAL);
''')
conn.commit()
print('Schema created.')

In [ ]:
# Synthetic sample data — illustrative only, not real business data
import random
random.seed(7)
# Populate a small, structurally representative sample so the queries below return results.
# (Replace with a real dataset, e.g. the UCI Online Retail set, to use this for real analysis.)
print('Populate sample rows here to match the schema above, then run the query cell below.')

In [ ]:
query = '''
WITH rfm_base AS (
  SELECT customer_id,
         MAX(invoice_date) AS last_order,
         COUNT(DISTINCT invoice_id) AS frequency,
         SUM(revenue) AS monetary
  FROM orders
  GROUP BY customer_id
),
rfm_scored AS (
  SELECT *,
         NTILE(5) OVER (ORDER BY last_order DESC) AS r_score,
         NTILE(5) OVER (ORDER BY frequency ASC) AS f_score,
         NTILE(5) OVER (ORDER BY monetary ASC) AS m_score
  FROM rfm_base
)
SELECT customer_id,
       ROUND(0.15*r_score + 0.28*f_score + 0.57*m_score, 2) AS rfm_score,
       CASE
         WHEN 0.15*r_score + 0.28*f_score + 0.57*m_score > 4.5 THEN 'Top Customer'
         WHEN 0.15*r_score + 0.28*f_score + 0.57*m_score > 4 THEN 'High-Value Customer'
         WHEN 0.15*r_score + 0.28*f_score + 0.57*m_score > 3 THEN 'Medium-Value Customer'
         WHEN 0.15*r_score + 0.28*f_score + 0.57*m_score > 1.6 THEN 'Low-Value Customer'
         ELSE 'Lost Customer'
       END AS tier
FROM rfm_scored
ORDER BY rfm_score DESC;
'''
for stmt in [s.strip() for s in query.split(';') if s.strip()]:
    try:
        cur.execute(stmt)
        rows = cur.fetchall()
        if rows:
            print(stmt.split(chr(10))[0], '...')
            for r in rows[:10]:
                print(r)
            print()
    except sqlite3.OperationalError as e:
        print('Skipped (SQLite dialect limitation):', e)